# Brain age on IXI: effect of a change of hospital (ClinicaDL 2.0)

**Before starting:** Runtime → Change runtime type → **T4 GPU**.

Organization:
- the scripts come from the GitHub repository [Ayoub-TAREK/ixi-brain-age](https://github.com/Ayoub-TAREK/ixi-brain-age) (`git clone` the first time, `git pull` afterwards);
- the data is downloaded to `/content` (fast, but erased at the end of the session);
- the results are saved on Drive (`MyDrive/test-clinicadl/outputs/`), so nothing is lost if the session disconnects.

Sections 1-12: v1 baseline (own 2 mm preprocessing) and its first augmentation runs. Sections 13-16: Clinica t1-linear images (v2), augmentation (v3a), AdaBN and DANN (v3b). For a one-click run of a source model + AdaBN + DANN on one hospital, see `notebooks/site_dann_pipeline.ipynb`.

## 1. Check the GPU

In [ ]:
!nvidia-smi
!python --version

## 2. Connecter Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Get the scripts from GitHub

Run again after each `git push` from the PC to get the latest version.

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/test-clinicadl'
WORK = '/content/test-clinicadl'
REPO = 'https://github.com/Ayoub-TAREK/ixi-brain-age.git'

# an old folder without .git (earlier version of the notebook) would block the clone
if os.path.exists(WORK) and not os.path.exists(f'{WORK}/.git'):
    !rm -rf {WORK}

if not os.path.exists(WORK):
    !git clone {REPO} {WORK}
else:
    !git -C {WORK} pull

os.makedirs(f'{WORK}/data/IXI/raw', exist_ok=True)
os.makedirs(f'{DRIVE}/outputs', exist_ok=True)
if not os.path.exists(f'{WORK}/outputs'):
    os.symlink(f'{DRIVE}/outputs', f'{WORK}/outputs')
!git -C {WORK} log --oneline -3
!ls {WORK}/scripts

## 3 bis. Unpack the t1-linear images (Clinica, 2 mm)

`caps_t1linear_2mm.tar` (1.5 GB, made on the PC by `scripts/t1_linear_wsl.sh`) is read from `MyDrive/test-clinicadl/` if it is on the mounted Drive. Otherwise it is downloaded with its Drive share link (about 1 min): paste the file ID (the part after `/d/` in the link) into `T1L_FILE_ID`. The file must be shared as *Anyone with the link*.

The images are unpacked into `data/IXI/caps_t1linear_2mm`; the cell is skipped if they are already there.

In [ ]:
import glob, os
T1L_FILE_ID = '1mdWBPxt_DrgNbCdlKAC_7HxAU7twkl3D'   # Drive share link ID of caps_t1linear_2mm.tar, used only if the .tar is not on the mounted Drive
T1L_TAR = f'{DRIVE}/caps_t1linear_2mm.tar'
T1L_DIR = f'{WORK}/data/IXI/caps_t1linear_2mm'
if not glob.glob(f'{T1L_DIR}/subjects/*/*/t1_linear/*_T1w.nii.gz'):
    os.makedirs(f'{WORK}/data/IXI', exist_ok=True)
    if os.path.exists(T1L_TAR):
        !tar -xf {T1L_TAR} -C {WORK}/data/IXI
    else:
        assert T1L_FILE_ID, f'{T1L_TAR} not found: set T1L_FILE_ID (Drive share link of the .tar)'
        !pip -q install gdown
        !gdown {T1L_FILE_ID} -O /content/caps_t1linear_2mm.tar
        !tar -xf /content/caps_t1linear_2mm.tar -C {WORK}/data/IXI && rm /content/caps_t1linear_2mm.tar
n = len(glob.glob(f'{T1L_DIR}/subjects/*/*/t1_linear/*_res-2x2x2_T1w.nii.gz'))
print(n, 't1-linear images (expected 563)')

## 4. Install the dependencies

In [ ]:
!pip install -q clinicadl==2.0.0 xlrd
!python -c "import clinicadl, torch; print('ClinicaDL', clinicadl.__version__, '| CUDA', torch.cuda.is_available())"

## 5. Download IXI (≈ 4.5 GB, a few minutes on Colab)

Official source: https://brain-development.org/ixi-dataset/. The step is skipped if the images are already there. Not needed for the t1-linear sections (13-16).

In [ ]:
import glob
%cd {WORK}/data/IXI
URL = 'http://biomedic.doc.ic.ac.uk/brain-development/downloads/IXI'
if not os.path.exists('IXI.xls'):
    !curl -fL --retry 3 -o IXI.xls {URL}/IXI.xls
if len(glob.glob('raw/*.nii.gz')) < 500:
    !curl -fL --retry 3 -o IXI-T1.tar {URL}/IXI-T1.tar
    !tar -xf IXI-T1.tar -C raw && rm IXI-T1.tar
!ls raw | wc -l

## 6. Conversion to BIDS + age table

In [ ]:
%cd {WORK}
!python scripts/ixi_to_bids.py

## Precomputed preprocessing (once per session, v1 only)

Resampling to 2 mm, normalization and cropping of all images into `data/IXI/bids_2mm`, so they are not recomputed at every epoch. Images already processed are skipped.

In [ ]:
%cd {WORK}
!python scripts/cache_preprocessed.py

## 7. Preprocessing check (one image per hospital)

In [ ]:
!python scripts/train_ixi_site_shift.py --check
from IPython.display import Image, display
for p in sorted(glob.glob(f'{WORK}/outputs/site_shift_test-IOP/check_*_sub-*.png')):
    display(Image(p))

## 7 bis. Archive run v1a and compute its signed predictions

To do **before** training again: the first run (v1a, MAE insite 6.16 / outofsite 12.21) is moved to `site_shift_test-IOP_v1_lr1e-3` so that `clean_run()` does not erase it, then its signed predictions are computed. The cell moves nothing if the archive already exists.

In [ ]:
import shutil
RUN_DIR = f'{WORK}/outputs/site_shift_test-IOP'
ARCHIVE = f'{WORK}/outputs/site_shift_test-IOP_v1_lr1e-3'

# The archiving step is done (v2 now lives in RUN_DIR): this cell no longer moves anything.
if os.path.exists(ARCHIVE):
    !python scripts/predict_ages.py --run-dir {ARCHIVE}
    for p in sorted(glob.glob(f'{ARCHIVE}/*.png')):
        if 'check_' not in p:
            display(Image(p))
else:
    print('No v1 archive found: nothing to do.')

## 8. Training: Guys + HH → test on IOP

Since run v1: the age is normalized inside the model and the learning rate is 1e-4 (change it with `--lr`).

ClinicaDL refuses to overwrite an existing MAPS or split folder: `clean_run()` deletes them before each launch (the check images are kept).

Start with `--epochs 2` to check, then launch the real training.

In [ ]:
import shutil
RUN_DIR = f'{WORK}/outputs/site_shift_test-IOP'

def clean_run():
    for name in ['maps', 'split', 'train_sites.tsv', 'out_of_site.tsv']:
        path = os.path.join(RUN_DIR, name)
        if os.path.isdir(path):
            shutil.rmtree(path)
        elif os.path.exists(path):
            os.remove(path)

clean_run()
!python scripts/train_ixi_site_shift.py --test-site IOP --epochs 2 --batch-size 8

In [ ]:
clean_run()
!python scripts/train_ixi_site_shift.py --test-site IOP --epochs 50 --batch-size 8

## 9. Results

MAE (in years) on the internal test set (`insite`, Guys + HH) and on the unseen hospital (`outofsite`, IOP).

In [ ]:
import pandas as pd
for group in ['insite', 'outofsite']:
    for tsv in sorted(glob.glob(f'{RUN_DIR}/maps/test/group-{group}/results/*/*/metrics/aggregated.tsv')):
        mae = pd.read_csv(tsv, sep='\t')['mae'].iloc[0]
        print(f'{group:12s} MAE = {mae:.2f} years')

## 10. Signed predictions (predicted − true age)

Reloads the `best-mae` model and computes the predicted age of every test subject: bias per age group, predicted vs true plot, validation curve. Everything is saved in the run folder.

In [ ]:
!python scripts/predict_ages.py --test-site IOP
for p in sorted(glob.glob(f'{RUN_DIR}/*.png')):
    if 'check_' not in p:
        display(Image(p))

## 11. Swap the unseen hospital (HH, then Guys)

Same experiment with another hospital held out: train on the two others, test on `TEST_SITE`. Results go to `outputs/site_shift_test-<site>`; the IOP run (`site_shift_test-IOP`) is never touched.

Run this cell once with `TEST_SITE = 'HH'`, then once with `TEST_SITE = 'Guys'` (~20-25 min each).

In [ ]:
import shutil
TEST_SITE = 'HH'   # then 'Guys'
SITE_DIR = f'{WORK}/outputs/site_shift_test-{TEST_SITE}'
assert TEST_SITE in ['HH', 'Guys'], 'This cell must never touch the IOP run'

# only cleans the folder of THIS test site (a previous run on the same site is replaced)
for name in ['maps', 'split', 'train_sites.tsv', 'out_of_site.tsv']:
    path = os.path.join(SITE_DIR, name)
    if os.path.isdir(path):
        shutil.rmtree(path)
    elif os.path.exists(path):
        os.remove(path)

!python scripts/train_ixi_site_shift.py --test-site {TEST_SITE} --epochs 50 --batch-size 8
!python scripts/predict_ages.py --test-site {TEST_SITE}
for p in sorted(glob.glob(f'{SITE_DIR}/predictions_*.png')) + glob.glob(f'{SITE_DIR}/validation_curve.png'):
    display(Image(p))

## 12. Same experiments with scanner-contrast augmentation (v3)

During training only, each image gets a random gamma (contrast between tissues, 0.70-1.42) and a smooth bias field, so the model learns to ignore each scanner's contrast style. Validation and test images are never augmented.

Results go to `outputs/site_shift_test-<site>_aug`, next to the v2 runs, which are never touched. Run this cell three times: `TEST_SITE = 'IOP'`, then `'HH'`, then `'Guys'` (~20-25 min each).

In [ ]:
import shutil
TEST_SITE = 'IOP'   # then 'HH', then 'Guys'
AUG_DIR = f'{WORK}/outputs/site_shift_test-{TEST_SITE}_aug'
assert AUG_DIR.endswith('_aug'), 'This cell must only touch *_aug folders'

# only cleans the *_aug folder of THIS test site (the v2 runs without augmentation are never touched)
for name in ['maps', 'split', 'train_sites.tsv', 'out_of_site.tsv']:
    path = os.path.join(AUG_DIR, name)
    if os.path.isdir(path):
        shutil.rmtree(path)
    elif os.path.exists(path):
        os.remove(path)

!python scripts/train_ixi_site_shift.py --test-site {TEST_SITE} --augment --epochs 50 --batch-size 8
!python scripts/predict_ages.py --test-site {TEST_SITE} --augment
for p in sorted(glob.glob(f'{AUG_DIR}/predictions_*.png')) + glob.glob(f'{AUG_DIR}/validation_curve.png'):
    display(Image(p))

## 13. Same experiments on Clinica t1-linear images

Images preprocessed with Clinica t1-linear (N4 bias correction + affine registration to MNI, then crop), unpacked by cell 3 bis. At load time they are padded to 96x112x96 and z-normalized.

For a session with **only** these runs, cells 2, 3, 3 bis, 4 and then this section are enough: no need to download IXI (cells 5-6) or to rebuild the cache.

Results go to `outputs/site_shift_test-<site>_t1linear` (and `..._t1linear_aug` with `AUGMENT = True`): the earlier runs are never touched. Run the training cell for `TEST_SITE = 'IOP'`, then `'HH'`, then `'Guys'`.

In [ ]:
import glob, os
from IPython.display import Image, display
%cd {WORK}
# one image per hospital: check the framing before training
!python scripts/train_ixi_site_shift.py --check --data t1linear
for p in sorted(glob.glob(f'{WORK}/outputs/site_shift_test-IOP_t1linear/check_*.png')):
    display(Image(p))

In [ ]:
import shutil
TEST_SITE = 'IOP'   # then 'HH', then 'Guys'
AUGMENT = False     # True: same run with scanner-contrast augmentation
T1_DIR = f"{WORK}/outputs/site_shift_test-{TEST_SITE}_t1linear{'_aug' if AUGMENT else ''}"
assert '_t1linear' in T1_DIR, 'This cell must only touch *_t1linear folders'
AUG_FLAG = '--augment' if AUGMENT else ''

# only cleans the folder of THIS run (the bids2mm runs are never touched)
for name in ['maps', 'split', 'train_sites.tsv', 'out_of_site.tsv']:
    path = os.path.join(T1_DIR, name)
    if os.path.isdir(path):
        shutil.rmtree(path)
    elif os.path.exists(path):
        os.remove(path)

!python scripts/train_ixi_site_shift.py --data t1linear --test-site {TEST_SITE} {AUG_FLAG} --epochs 50 --batch-size 8
!python scripts/predict_ages.py --data t1linear --test-site {TEST_SITE} {AUG_FLAG}
for p in sorted(glob.glob(f'{T1_DIR}/predictions_*.png')) + glob.glob(f'{T1_DIR}/validation_curve.png'):
    display(Image(p))

### Summary of all runs

In-site / out-of-site MAE (years) of every finished run, read from the MAPS test results.

In [ ]:
import pandas as pd
rows = []
for run in sorted(glob.glob(f'{WORK}/outputs/site_shift_test-*')):
    row = {'run': os.path.basename(run).removeprefix('site_shift_test-')}
    for group in ['insite', 'outofsite']:
        tsvs = glob.glob(f'{run}/maps/test/group-{group}/results/*/*/metrics/aggregated.tsv')
        if tsvs:
            row[group] = round(pd.read_csv(tsvs[0], sep='\t')['mae'].iloc[0], 2)
    if len(row) > 1:
        rows.append(row)
pd.DataFrame(rows)

## 14. AdaBN (v3b): adapt the trained models to the held-out hospital

No training: for each model trained in section 13, the BatchNorm statistics are re-estimated on the images of the held-out hospital (its ages are never used), then its subjects are predicted again. About 1-2 min per model.

Needs cells 2, 3, 3 bis and 4, and the finished runs of section 13 (`*_t1linear` and `*_t1linear_aug` folders on Drive). Outputs: `predictions_adabn_best-mae.tsv` / `.png` in each run folder.

In [ ]:
import glob, os
import pandas as pd
from IPython.display import Image, display
%cd {WORK}
for site in ['IOP', 'HH', 'Guys']:
    for aug in ['', '--augment']:
        run = f"{WORK}/outputs/site_shift_test-{site}_t1linear{'_aug' if aug else ''}"
        if os.path.exists(f'{run}/maps/training/split-0/models/best_models/best-mae/model.pt'):
            print(f'\n===== {os.path.basename(run)} =====')
            !python scripts/adabn.py --data t1linear --test-site {site} {aug} 2>&1 | grep -v -i warning | tail -n 22
        else:
            print(f'skipped (no trained model): {run}')

In [ ]:
# out-of-site MAE / bias / r before and after AdaBN, for every run
rows = []
for tsv in sorted(glob.glob(f'{WORK}/outputs/site_shift_test-*_t1linear*/predictions_adabn_best-mae.tsv')):
    df = pd.read_csv(tsv, sep='\t')
    row = {'run': os.path.basename(os.path.dirname(tsv)).removeprefix('site_shift_test-')}
    for group, name in [('outofsite', 'before'), ('outofsite_adabn', 'adabn')]:
        d = df[df.group == group]
        row[f'MAE {name}'] = round(d.error.abs().mean(), 2)
        row[f'bias {name}'] = round(d.error.mean(), 2)
        row[f'r {name}'] = round(d.AgeMRI.corr(d.predicted), 2)
    rows.append(row)
pd.DataFrame(rows)

## 15. Stabilized training, several seeds

Same as section 13, plus a learning-rate scheduler (`--scheduler plateau`: the learning rate is halved when the validation MAE has not improved for 3 epochs; early stopping waits 15 epochs instead of 10), and one run per seed. The data split stays the same for every seed (seed 42), so all runs are tested on the same subjects: only the training randomness changes (initialization, batch order, augmentation).

Each run is followed by `predict_ages` and AdaBN, on the **final model** (the model at the end of training). With noisy validation curves, the best validation epoch is often an early lucky dip: on IOP (3 seeds) the final model is 1.5 years better out-of-site, for every seed, with the same in-site MAE. Results go to `outputs/site_shift_test-<site>_t1linear[_aug]_plateau[_seed<n>]`: earlier runs are never touched. Augmented runs are faster than in section 13 (bias field computed on a coarse grid).

Suggested first step: `TEST_SITE = 'IOP'`, `AUGMENT = True`, 3 seeds (about 1.5-2 h of GPU). Already finished runs are skipped, so the cell can be rerun after a disconnection.

In [ ]:
import glob, os, shutil
from IPython.display import Image, display
%cd {WORK}
TEST_SITE = 'IOP'      # 'IOP', 'HH' or 'Guys'
AUGMENT = True
SEEDS = [42, 1, 2]

for seed in SEEDS:
    run = (f"{WORK}/outputs/site_shift_test-{TEST_SITE}_t1linear{'_aug' if AUGMENT else ''}_plateau"
           + ('' if seed == 42 else f'_seed{seed}'))
    assert '_plateau' in run, 'This cell must only touch *_plateau folders'
    flags = f"--data t1linear --test-site {TEST_SITE} --scheduler plateau --seed {seed} {'--augment' if AUGMENT else ''}"
    if os.path.exists(f'{run}/predictions_adabn_final.tsv'):
        print(f'already done, skipped: {os.path.basename(run)}')
        continue
    # unfinished run (e.g. disconnection): start it again from scratch
    for name in ['maps', 'split', 'train_sites.tsv', 'out_of_site.tsv']:
        path = os.path.join(run, name)
        if os.path.isdir(path):
            shutil.rmtree(path)
        elif os.path.exists(path):
            os.remove(path)
    print(f'\n===== {os.path.basename(run)} =====')
    !python scripts/train_ixi_site_shift.py {flags} --epochs 50 --batch-size 8 2>&1 | grep -E "seed|scheduler|stopped|Error|Traceback"
    !python scripts/predict_ages.py {flags} 2>&1 | grep -v -i warning | tail -n 12
    !python scripts/adabn.py {flags} --checkpoint final 2>&1 | grep -v -i warning | tail -n 7
    display(Image(f'{run}/validation_curve.png'))

In [ ]:
# final models: MAE per run, then mean and spread over the seeds of each configuration
import re
import pandas as pd
rows = []
for tsv in sorted(glob.glob(f'{WORK}/outputs/site_shift_test-*_plateau*/predictions_adabn_final.tsv')):
    name = os.path.basename(os.path.dirname(tsv)).removeprefix('site_shift_test-')
    df = pd.read_csv(tsv, sep='\t')
    row = {'config': re.sub(r'_seed\d+$', '', name),
           'seed': int(m.group(1)) if (m := re.search(r'_seed(\d+)$', name)) else 42}
    for group, col in [('insite', 'insite'), ('outofsite', 'outofsite'), ('outofsite_adabn', 'adabn')]:
        d = df[df.group == group]
        row[f'MAE {col}'] = d.error.abs().mean()
        row[f'bias {col}'] = d.error.mean()
    rows.append(row)
runs = pd.DataFrame(rows)
display(runs.round(2))
runs.drop(columns='seed').groupby('config').agg(['mean', 'std']).round(2)

## 16. DANN (v3b): domain-adversarial fine-tuning

Starts from the final model of a finished run of section 15 (same `TEST_SITE`, augmentation and seed) and fine-tunes it for 30 epochs: the age head keeps learning on the training hospitals, while a domain classifier, connected through a gradient reversal layer, pushes the features to make the held-out hospital indistinguishable (its ages are never used). Details and choices: docstring of `scripts/train_dann.py` and `docs/journal.md`.

Configurations, as `(lambda_max, iw)`:
- `(0, False)`: control, same fine-tuning without the domain loss (separates the effect of DANN from the effect of 30 more epochs);
- `(0.1, False)`, `(0.3, False)`: standard DANN, two alignment strengths;
- `(0.3, True)`: IW-DANN, source images re-weighted by age to match the (predicted) age distribution of the target, against the label shift that made AdaBN fail on IOP.

Honest protocol: target ages must not be used to choose the configuration. All configurations are reported (sensitivity), and the main configuration is fixed in advance: IW-DANN, lambda 0.3.

About 30 min per configuration on a T4. Finished configurations are skipped.

In [ ]:
import glob, os
from IPython.display import Image, display
%cd {WORK}
TEST_SITE = 'IOP'
AUGMENT = True
SEED = 42
CONFIGS = [(0, False), (0.1, False), (0.3, False), (0.3, True)]   # (lambda_max, iw)

source = f"--data t1linear --test-site {TEST_SITE} --scheduler plateau --seed {SEED} {'--augment' if AUGMENT else ''}"
src_dir = (f"{WORK}/outputs/site_shift_test-{TEST_SITE}_t1linear{'_aug' if AUGMENT else ''}_plateau"
           + ('' if SEED == 42 else f'_seed{SEED}'))
assert os.path.exists(f'{src_dir}/predictions_adabn_final.tsv'), f'run section 15 first: {src_dir}'
for lam, iw in CONFIGS:
    run = f"{src_dir}_dann{lam:g}{'_iw' if iw else ''}"
    if os.path.exists(f'{run}/predictions_final.tsv'):
        print(f'already done, skipped: {os.path.basename(run)}')
        continue
    print(f'\n===== {os.path.basename(run)} =====')
    !python scripts/train_dann.py {source} --lambda-max {lam} {'--iw' if iw else ''} 2>&1 | grep -v -i warning
    !python scripts/predict_ages.py --run-dir {run} --data t1linear --checkpoint final 2>&1 | grep -v -i warning | tail -n 12
    display(Image(f'{run}/dann_curves.png'))

In [ ]:
# out-of-site results of the source model and of each DANN configuration (final models everywhere)
import pandas as pd
def scores(tsv, name):
    df = pd.read_csv(tsv, sep='\t')
    d, s = df[df.group == 'outofsite'], df[df.group == 'insite']
    return {'run': name, 'MAE insite': s.error.abs().mean(), 'MAE outofsite': d.error.abs().mean(),
            'bias outofsite': d.error.mean(), 'r outofsite': d.AgeMRI.corr(d.predicted)}
rows = []
for src in sorted(glob.glob(f'{WORK}/outputs/site_shift_test-*_plateau*/predictions_adabn_final.tsv')):
    src_dir = os.path.dirname(src)
    if '_dann' in src_dir:
        continue
    rows.append(scores(src, os.path.basename(src_dir).removeprefix('site_shift_test-') + ' (source model)'))
    for tsv in sorted(glob.glob(f'{src_dir}_dann*/predictions_final.tsv')):
        name = os.path.basename(os.path.dirname(tsv))
        row = scores(tsv, '  ' + name[name.index('_dann') + 1:])
        log = pd.read_csv(os.path.join(os.path.dirname(tsv), 'dann_log.tsv'), sep='\t')
        row['domain acc (last)'] = log.domain_acc.iloc[-1]
        rows.append(row)
pd.DataFrame(rows).round(2)